# 1단계 : 리뷰 텍스트 기본 전처리

## 목적
무신사 리뷰 텍스트를 임베딩/토픽 모델링/감성분석에 사용할 수 있도록 노이즈를 제거하고, 분석 단계별로 사용할 컬럼을 생성한다.

## 입력 / 출력
- **입력**: `cleaned_reviews.csv` (685,292건)
- **출력**: `reviews_step1_cleaned.csv`

## 생성 컬럼 요약
| 컬럼 | 용도 | 처리 강도 |
|---|---|---|
| `리뷰내용` | 원본 보존 (수정 X) | - |
| `리뷰내용_clean` | 임베딩, 토픽 모델링용 | 약한 정제 |
| `리뷰내용_norm` | 중복 검출용 (비교 키) | 강한 정규화 |
| `laugh_count`, `cry_count`, `exclamation_count`, `question_count`, `emoji_count`, `has_repetition`, `text_length_orig` | 감정 강도 플래그 | 원본에서 추출 |
| `is_valid_for_topic` | 토픽 모델링 입력 여부 (한글 5자 이상) | - |

## 처리 절차
1. 데이터 로드
2. 노이즈 패턴 확인
3. 정제 함수 정의
4. 정제 함수 적용
5. 감정 강도 플래그 추출
6. 유효 리뷰 플래그 생성
7. 전후 비교 검증
8. 빈 리뷰 확인
9. 저장

## 1. 데이터 로드

원본 CSV를 불러오고, 리뷰 본문이 NaN인 행을 제거한다.

In [1]:
import re
import numpy as np
import pandas as pd
import emoji
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore', 
                        message='This pattern is interpreted as a regular expression')

CSV_PATH = "cleaned_reviews.csv"
TEXT_COL = "리뷰내용"

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"로드 완료: {len(df):,}건")

# 리뷰 본문 NaN 제거
before = len(df)
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
print(f"NaN 제거: {before:,} → {len(df):,}건 (-{before-len(df):,})")

로드 완료: 685,292건
NaN 제거: 685,292 → 685,042건 (-250)


## 2. 노이즈 패턴 확인

정제 전, 데이터에 어떤 노이즈 패턴이 얼마나 분포하는지 확인하여 정제 대상을 파악한다.

In [2]:
# 노이즈 패턴 분포 확인
sample = df[TEXT_COL].astype(str).astype('object')

print("[노이즈 패턴 분포]")
print(f"엑셀 오류값(#NAME? 등): {sample.str.contains(r'#(?:NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)', regex=True, na=False).sum():,}건")
print(f"HTML 태그(<br> 등):     {sample.str.contains(r'<[^>]+>', regex=True, na=False).sum():,}건")
print(f"HTML 엔티티(&nbsp;):    {sample.str.contains(r'&[a-z]+;', regex=True, na=False).sum():,}건")
print(f"URL:                  {sample.str.contains(r'https?://|www\.', regex=True, na=False).sum():,}건")
print(f"줄바꿈/탭:             {sample.str.contains(r'[\r\n\t]', regex=True, na=False).sum():,}건")
print(f"반복문자 4회+:          {sample.str.contains(r'(.)\1{3,}', regex=True, na=False).sum():,}건")
print(f"이모지 포함:           {sample.apply(lambda x: emoji.emoji_count(x) > 0).sum():,}건")
print(f"한글 없음:             {(~sample.str.contains(r'[가-힣]', regex=True, na=False)).sum():,}건")

[노이즈 패턴 분포]
엑셀 오류값(#NAME? 등): 0건
HTML 태그(<br> 등):     234건
HTML 엔티티(&nbsp;):    2,167건
URL:                  5건
줄바꿈/탭:             0건
반복문자 4회+:          5,994건
이모지 포함:           7,036건
한글 없음:             184건


## 3. 정제 함수 정의

목적이 다른 두 가지 정제 함수를 정의한다.

- **`clean_review_text`** : 약한 정제. 의미를 보존하면서 노이즈만 제거. 임베딩과 토픽 모델링에 사용.
- **`normalize_review_text`** : 강한 정규화. 중복 검출 시 동일 리뷰를 같은 키로 매핑하기 위한 용도.

> **`리뷰내용_norm`에서 `%`, `/` 같은 기호를 모두 제거하는 이유**
> 이 컬럼은 사람이 읽거나 분석에 쓰는 텍스트가 아니라 "이 두 리뷰가 같은가?"를 비교하는 **지문(fingerprint)** 역할이다. 같은 내용을 약간 다른 형식으로 쓴 리뷰들이 동일하게 매칭되어야 중복 검출이 정확해진다.
> 의미 정보는 `리뷰내용_clean`에 그대로 보존되어 있다.

In [3]:
def clean_review_text(text):
    """
    약한 정제 (임베딩, 토픽 모델링용).
    의미는 보존하면서 노이즈만 제거.
    """
    text = str(text)
    
    # (1) 엑셀 오류값
    text = re.sub(r'#(?:NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)[!?]?', ' ', text)
    
    # (2) URL
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    # (3) HTML 태그 + 엔티티
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    
    # (4) 줄바꿈/탭 → 공백
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # (5) 이모지 제거 (감성 신호는 emoji_count 플래그로 별도 보존)
    text = emoji.replace_emoji(text, replace=' ')
    
    # (6) 반복문자 정규화 (약하게: 4회+ → 3회로 압축)
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1{3,}', r'\1\1\1', text)   # 자모
    text = re.sub(r'(.)\1{3,}', r'\1\1\1', text)            # 일반 문자
    
    # (7) 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# 셀 3의 normalize_review_text 함수 교체
def normalize_review_text(text):
    """
    강한 정규화 (중복 검출 전용).
    자모만 압축하고 숫자/영문/한글 완성형은 보존.
    """
    text = str(text)
    
    # 기본 노이즈 제거
    text = re.sub(r'#(?:NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)[!?]?', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&[a-zA-Z]+;', '', text)
    
    # 이모지 완전 제거
    text = emoji.replace_emoji(text, replace='')
    
    # ⚠️ 수정: 자모만 1개로 압축 (숫자/영문/한글 완성형은 보존)
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1+', r'\1', text)
    
    # 한글/영문/숫자만 남기고 모두 제거 (공백, 문장부호 포함)
    text = re.sub(r'[^가-힣a-zA-Z0-9]', '', text)
    
    return text

print("정제 함수 정의 완료")


정제 함수 정의 완료


`%나 /같은 기호를 빼는 이유!` : 리뷰내용_norm은 중복 검출 전용 컬럼이기 때문.

## 4. 정제 함수 적용

두 함수를 데이터프레임 전체에 적용해 `리뷰내용_clean`, `리뷰내용_norm` 컬럼을 생성한다.

In [4]:
tqdm.pandas()

df['리뷰내용_clean'] = df[TEXT_COL].progress_apply(clean_review_text)
df['리뷰내용_norm']  = df[TEXT_COL].progress_apply(normalize_review_text)

print("정제 완료")
print(f"  리뷰내용_clean: 약한 정제 (임베딩/토픽 모델링용)")
print(f"  리뷰내용_norm : 강한 정규화 (중복 검출용)")

  0%|          | 0/685042 [00:00<?, ?it/s]

  0%|          | 0/685042 [00:00<?, ?it/s]

정제 완료
  리뷰내용_clean: 약한 정제 (임베딩/토픽 모델링용)
  리뷰내용_norm : 강한 정규화 (중복 검출용)


## 5. 감정 강도 플래그 추출 (원본 기준)

정제 과정에서 압축되는 감정 신호(`ㅋㅋㅋㅋ`, `!!!`, 이모지 등)의 강도를 별도 컬럼으로 보존한다.
**원본 텍스트** 기준으로 추출해야 정확한 강도가 측정된다 (정제 후엔 이미 압축되어 있음).

| 플래그 | 의미 |
|---|---|
| `laugh_count` | ㅋ/ㅎ 연속 최대 길이 |
| `cry_count` | ㅠ/ㅜ 연속 최대 길이 |
| `exclamation_count` | `!` 개수 |
| `question_count` | `?` 개수 |
| `emoji_count` | 이모지 개수 |
| `has_repetition` | 반복문자 존재 여부 (bool) |
| `text_length_orig` | 원본 길이 |

In [5]:
def extract_flags(text):
    """원본 텍스트에서 감정 강도/통계 플래그 추출."""
    text = str(text)
    
    # ㅋ/ㅎ 연속 최대 길이
    laugh_matches = re.findall(r'[ㅋㅎ]+', text)
    laugh_count = max((len(m) for m in laugh_matches), default=0)
    
    # ㅠ/ㅜ 연속 최대 길이
    cry_matches = re.findall(r'[ㅠㅜ]+', text)
    cry_count = max((len(m) for m in cry_matches), default=0)
    
    return pd.Series({
        'laugh_count':       laugh_count,
        'cry_count':         cry_count,
        'exclamation_count': text.count('!'),
        'question_count':    text.count('?'),
        'emoji_count':       emoji.emoji_count(text),
        'has_repetition':    bool(re.search(r'(.)\1{2,}', text)),
        'text_length_orig':  len(text),
    })

# 원본 텍스트 기준으로 플래그 추출
flags = df[TEXT_COL].progress_apply(extract_flags)
df = pd.concat([df, flags], axis=1)

print("플래그 추출 완료")
print(df[['laugh_count','cry_count','exclamation_count',
          'question_count','emoji_count','has_repetition',
          'text_length_orig']].describe().round(2))

  0%|          | 0/685042 [00:00<?, ?it/s]

플래그 추출 완료
       laugh_count  cry_count  exclamation_count  question_count  emoji_count  \
count    685042.00  685042.00          685042.00       685042.00    685042.00   
mean          0.11       0.05               0.30            0.02         0.02   
std           0.49       0.30               0.81            0.18         0.26   
min           0.00       0.00               0.00            0.00         0.00   
25%           0.00       0.00               0.00            0.00         0.00   
50%           0.00       0.00               0.00            0.00         0.00   
75%           0.00       0.00               0.00            0.00         0.00   
max          53.00      19.00              40.00           11.00        80.00   

       text_length_orig  
count         685042.00  
mean              44.93  
std               30.69  
min                1.00  
25%               29.00  
50%               36.00  
75%               49.00  
max             1460.00  


## 6. 유효 리뷰 플래그

한글 5자 미만 리뷰는 토픽 모델링에 의미가 없어 제외 대상으로 표시한다.
실제 행은 삭제하지 않고 `is_valid_for_topic` 플래그로만 구분하여, 이후 단계에서 선택적으로 필터링할 수 있게 한다.

In [6]:
df['한글_글자수']  = df['리뷰내용_clean'].str.count(r'[가-힣]')
df['전체_글자수']  = df['리뷰내용_clean'].str.len()
df['is_valid_for_topic'] = df['한글_글자수'] >= 5

print(f"[유효 리뷰 분포]")
print(f"  유효 (한글 5자 이상): {df['is_valid_for_topic'].sum():,}건 "
      f"({df['is_valid_for_topic'].mean()*100:.2f}%)")
print(f"  무효 (한글 5자 미만): {(~df['is_valid_for_topic']).sum():,}건")

print("\n[무효로 분류된 리뷰 샘플]")
display(df.loc[~df['is_valid_for_topic'], 
               ['리뷰내용', '리뷰내용_clean', '한글_글자수']].head(10))

[유효 리뷰 분포]
  유효 (한글 5자 이상): 684,747건 (99.96%)
  무효 (한글 5자 미만): 295건

[무효로 분류된 리뷰 샘플]


,리뷰내용,리뷰내용_clean,한글_글자수
5451,어어어어어어어어어어어어어어어어어어어엉,어어어엉,4
14184,采用优质棉质面料，吸汗透气舒适，大小合适，买到合体的衣服，非常高兴！,采用优质棉质面料，吸汗透气舒适，大小合适，买到合体的衣服，非常高兴！,0
15368,솦호호56)?56&))₩₩(6)((5)(tygfyhdryhgfh,솦호호56)?56&))₩₩(6)((5)(tygfyhdryhgfh,3
16593,굳,굳,1
21129,굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿굿,굿굿굿,3
31198,whehudisjwndnckcohxhsn,whehudisjwndnckcohxhsn,0
35450,12345567889909765543호ㅓㅗ포ㅜ,12345567889909765543호ㅓㅗ포ㅜ,2
37415,GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD,GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD GOOD,0
43875,14163662626277262671,14163662626277262671,0
49392,goo1oo1oo1o1oo1o1ood!,goo1oo1oo1o1oo1o1ood!,0


## 7. 전후 비교 검증

정제가 의도대로 동작했는지 샘플로 확인한다. 원본과 두 정제 컬럼을 나란히 비교한다.

In [7]:
# 정제 전후 비교
print("[정제 전후 비교 샘플]")
sample_idx = df[df['리뷰내용'] != df['리뷰내용_clean']].sample(10, random_state=42).index
for i in sample_idx:
    print(f"\n리뷰번호: {df.loc[i, '리뷰번호']}")
    print(f"  원본       : {repr(df.loc[i, '리뷰내용'])[:100]}")
    print(f"  clean      : {repr(df.loc[i, '리뷰내용_clean'])[:100]}")
    print(f"  norm(중복용): {repr(df.loc[i, '리뷰내용_norm'])[:100]}")

print("\n[글자 수 통계]")
print(df[['전체_글자수', '한글_글자수']].describe())

changed = (df['리뷰내용'].astype(str) != df['리뷰내용_clean']).sum()
print(f"\n정제로 인해 변경된 리뷰: {changed:,}건 ({changed/len(df)*100:.1f}%)")

[정제 전후 비교 샘플]

리뷰번호: 63376867
  원본       : '착용감: 면 100%로 부드럽습니다. 여름에 입기에 적당한 두께로 시원하게 착용하기 좋습니다.   사이즈: 188/74의 스펙인데 오버핏보다는 세미 오버핏이 나옵니다.  언제: 
  clean      : '착용감: 면 100%로 부드럽습니다. 여름에 입기에 적당한 두께로 시원하게 착용하기 좋습니다. 사이즈: 188/74의 스펙인데 오버핏보다는 세미 오버핏이 나옵니다. 언제: 여름에
  norm(중복용): '착용감면100로부드럽습니다여름에입기에적당한두께로시원하게착용하기좋습니다사이즈18874의스펙인데오버핏보다는세미오버핏이나옵니다언제여름에입기에좋은원단과프린팅입니다단품으로입기에좋은반팔티입

리뷰번호: 53631187
  원본       : '딱 기본 후드티네요 좋아요 안에 입기 좋습니다 추천해요 '
  clean      : '딱 기본 후드티네요 좋아요 안에 입기 좋습니다 추천해요'
  norm(중복용): '딱기본후드티네요좋아요안에입기좋습니다추천해요'

리뷰번호: 42243407
  원본       : '가성비 갑!!  배송도 빠르고 옷도 괜찮아요 추천합니다~'
  clean      : '가성비 갑!! 배송도 빠르고 옷도 괜찮아요 추천합니다~'
  norm(중복용): '가성비갑배송도빠르고옷도괜찮아요추천합니다'

리뷰번호: 60388747
  원본       : '무신사에서 입어보고 핏 갠찬아서 삿어요 팔소매 걷어서 입으면 이쁩니니다  차콜 반팔티로 추천!'
  clean      : '무신사에서 입어보고 핏 갠찬아서 삿어요 팔소매 걷어서 입으면 이쁩니니다 차콜 반팔티로 추천!'
  norm(중복용): '무신사에서입어보고핏갠찬아서삿어요팔소매걷어서입으면이쁩니니다차콜반팔티로추천'

리뷰번호: 61725743
  원본       : '넉넉하고 도톰한원단이라 간절기 아우터로도 좋을거 같아요! 색상도  이뻐요'
  clean      : '넉넉하고 도톰한원단이라 간절기 아우터로도 좋을거 같아요! 

## 8. 빈 리뷰 확인

정제 후 텍스트가 비어버린 리뷰를 확인한다. 일반적으로 URL만 있던 리뷰나 이모지로만 구성된 리뷰가 해당된다.
이런 행도 `is_valid_for_topic = False`로 자동 분류되어 토픽 모델링에서는 제외된다.

In [8]:
empty_after = df[df['전체_글자수'] == 0]
print(f"정제 후 빈 리뷰: {len(empty_after):,}건")
print("\n[원본 확인]")
display(empty_after[['리뷰번호', '리뷰내용']].head(20))

정제 후 빈 리뷰: 5건

[원본 확인]


,리뷰번호,리뷰내용
279991,24482388,https://youtu.be/8ebPs7hKrc4
493929,66155353,👍🏻👍🏻
529051,66208045,🖤
529052,66206367,🤍🖤
549484,66197514,👍


## 9. 저장

전처리 결과를 `reviews_step1_cleaned.csv`로 저장한다. 다음 단계(중복 처리)에서 이 파일을 입력으로 사용한다.

In [9]:
output_path = "./reviews_step1_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"저장 완료: {output_path}")
print(f"\n[저장된 컬럼 요약]")
print(f"  텍스트 컬럼: 리뷰내용(원본), 리뷰내용_clean, 리뷰내용_norm")
print(f"  플래그    : laugh_count, cry_count, exclamation_count,")
print(f"             question_count, emoji_count, has_repetition, text_length_orig")
print(f"  유효성    : 한글_글자수, 전체_글자수, is_valid_for_topic")
print(f"\n전체 행 수: {len(df):,}건")

저장 완료: ./reviews_step1_cleaned.csv

[저장된 컬럼 요약]
  텍스트 컬럼: 리뷰내용(원본), 리뷰내용_clean, 리뷰내용_norm
  플래그    : laugh_count, cry_count, exclamation_count,
             question_count, emoji_count, has_repetition, text_length_orig
  유효성    : 한글_글자수, 전체_글자수, is_valid_for_topic

전체 행 수: 685,042건
